In [1]:
!pip install -q transformers torch sentencepiece

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading summarization model...")

summary_model_name = "facebook/bart-large-cnn"
summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    summary_model_name
).to(device)

print("Loading question-answering model...")

qa_model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
).to(device)

summary_model.eval()
qa_model.eval()

print("Models loaded successfully!")
print("Device:", device)

text = """
Artificial Intelligence is transforming many industries by enabling
machines to perform tasks that normally require human intelligence.
It is widely used in healthcare, education, manufacturing, finance,
transportation, and cybersecurity. AI systems can analyze large
amounts of data, identify patterns, make predictions, and support
intelligent decision-making. Generative AI is a branch of Artificial
Intelligence that can create new content such as text, images, audio,
video, and computer programs.
"""

summary_inputs = summary_tokenizer(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

with torch.no_grad():
    summary_ids = summary_model.generate(
        **summary_inputs,
        max_new_tokens=60,
        min_new_tokens=20,
        num_beams=4,
        do_sample=False,
        early_stopping=True
    )

summary = summary_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("\nTEXT SUMMARIZATION")
print("-" * 60)
print("Original Text:")
print(text.strip())
print("\nSummary:")
print(summary)

context = """
Generative Artificial Intelligence is a type of Artificial Intelligence
that can create new content such as text, images, audio, video, and
computer programs. Large Language Models are commonly used for text
generation, summarization, translation, and question answering.
"""

question = "What type of content can Generative AI create?"

qa_inputs = qa_tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    qa_outputs = qa_model(**qa_inputs)

start_index = torch.argmax(qa_outputs.start_logits, dim=1).item()
end_index = torch.argmax(qa_outputs.end_logits, dim=1).item()

if end_index < start_index:
    end_index = start_index

answer_tokens = qa_inputs["input_ids"][0][start_index:end_index + 1]

answer = qa_tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)

start_probability = torch.softmax(
    qa_outputs.start_logits,
    dim=1
)[0][start_index]

end_probability = torch.softmax(
    qa_outputs.end_logits,
    dim=1
)[0][end_index]

confidence = torch.sqrt(
    start_probability * end_probability
).item()

print("\nQUESTION ANSWERING")
print("-" * 60)
print("Context:")
print(context.strip())
print("\nQuestion:", question)
print("Answer:", answer)
print("Confidence Score:", round(confidence, 3))

Loading summarization model...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Models loaded successfully!
Device: cpu


[transformers] Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEXT SUMMARIZATION
------------------------------------------------------------
Original Text:
Artificial Intelligence is transforming many industries by enabling
machines to perform tasks that normally require human intelligence.
It is widely used in healthcare, education, manufacturing, finance,
transportation, and cybersecurity. AI systems can analyze large
amounts of data, identify patterns, make predictions, and support
intelligent decision-making. Generative AI is a branch of Artificial
Intelligence that can create new content such as text, images, audio,
video, and computer programs.

Summary:
Artificial Intelligence is transforming many industries by enabling machines to perform tasks that normally require human intelligence. It is widely used in healthcare, education, manufacturing, finance,transportation, and cybersecurity.

QUESTION ANSWERING
------------------------------------------------------------
Context:
Generative Artificial Intelligence is a type of Artificial Inte